In [1]:
# Parameters
session_id = "20250927_203811"


In [2]:
# Parameters (for papermill)
session_id = None  # Will be injected by papermill or default to latest

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import json
import os
import pickle  # Added for PKL generation
from pipeline import train_and_evaluate_models  # Import training function

def load_streamlit_session(session_id=None):
    try:
        if session_id is None:
            # Get latest session ID if not provided
            with open("latest_session.txt", "r") as f:
                session_id = f.read().strip()
        
        session_dir = f"session_{session_id}"
        
        # Load session data
        with open(os.path.join(session_dir, "session_data.json"), "r") as f:
            session_data = json.load(f)
        
        # Load dataframe
        df = pd.read_pickle(session_data['dataframe_path'])
        
        return df, session_data, session_dir
    
    except FileNotFoundError as e:
        print(f"❌ Session data not found: {e}")
        print("Please upload a file through Streamlit first.")
        return None, None, None

# Load data from Streamlit session
df, config, session_dir = load_streamlit_session(session_id)

if df is not None and config is not None:
    print("✅ Successfully loaded data from Streamlit!")
    print(f"📊 Dataset shape: {df.shape}")
    print(f"🎯 Target column: {config['y_col']}")
    print(f"📈 Features: {config['X_cols']}")
    print(f"🔄 Task type: {config['task_type']}")
    
    # Now you can work with the data
    X = df[config['X_cols']]
    y = df[config['y_col']]
    
    print(f"\n📋 Feature matrix shape: {X.shape}")
    print(f"📋 Target vector shape: {y.shape}")
    
    df.head()
else:
    print("Please run 'streamlit run app.py' and upload a file first.")

✅ Successfully loaded data from Streamlit!
📊 Dataset shape: (80000, 5)
🎯 Target column: Goal
📈 Features: ['Gender', 'BMI Category', 'Exercise Schedule', 'Meal Plan']
🔄 Task type: Classification

📋 Feature matrix shape: (80000, 4)
📋 Target vector shape: (80000,)


In [4]:
df.head()

,Gender,Goal,BMI Category,Exercise Schedule,Meal Plan
0,Female,muscle_gain,Normal weight,"Moderate cardio, Strength training, and 5000 s...",Balanced diet with moderate protein and carboh...
1,Male,fat_burn,Underweight,"Light weightlifting, Yoga, and 2000 steps walking","High-calorie, protein-rich diet: Whole milk, p..."
2,Male,muscle_gain,Normal weight,"Moderate cardio, Strength training, and 5000 s...",Balanced diet with moderate protein and carboh...
3,Male,muscle_gain,Overweight,"High-intensity interval training (HIIT), Cardi...","Low-carb, high-fiber diet: Avocado, grilled fi..."
4,Female,muscle_gain,Normal weight,"Moderate cardio, Strength training, and 5000 s...",Balanced diet with moderate protein and carboh...


In [5]:
# Count missing values before
missing_before = df.isnull().sum().sum()

# Copy X and y
X = df[config['X_cols']].copy()
y = df[config['y_col']].copy()

# Fill missing values in X
for col in X.columns:
    if X[col].dtype in ["int64", "float64"]:  # numeric
        X[col] = X[col].fillna(X[col].mean())
    else:  # categorical/text
        X[col] = X[col].fillna(X[col].mode()[0])

# Fill missing values in y
if y.isnull().sum() > 0:
    if y.dtype in ["int64", "float64"]:
        y = y.fillna(y.mean())
    else:
        y = y.fillna(y.mode()[0])

# Calculate missing after
missing_after = pd.concat([X, pd.DataFrame(y)], axis=1).isnull().sum().sum()

print(f"✅ Missing values handled: {missing_before - missing_after}")

✅ Missing values handled: 0


In [6]:
from sklearn.preprocessing import LabelEncoder

encoders = {}

# Encode categorical X columns
for col in X.columns:
    if X[col].dtype == "object" or str(X[col].dtype) == "category":
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        encoders[col] = le

# Encode target if classification
if config['task_type'] == "classification" and (
    y.dtype == "object" or str(y.dtype) == "category"
):
    le = LabelEncoder()
    y = le.fit_transform(y.astype(str))
    encoders[config['y_col']] = le

print("✅ Encoding complete. Encoded categorical columns:", list(encoders.keys()))

✅ Encoding complete. Encoded categorical columns: ['Gender', 'BMI Category', 'Exercise Schedule', 'Meal Plan']


In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)

print("✅ Features scaled")

✅ Features scaled


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("✅ Train/Test split complete")

✅ Train/Test split complete


In [9]:
print("🔎 Preprocessing Summary")
print("Rows before preprocessing:", df.shape[0])
print("Rows after preprocessing:", X_train.shape[0] + X_test.shape[0])
print("Missing values handled:", missing_before - missing_after)
print("Final Train shape:", X_train.shape, "Target:", y_train.shape)
print("Final Test shape:", X_test.shape, "Target:", y_test.shape)

🔎 Preprocessing Summary
Rows before preprocessing: 80000
Rows after preprocessing: 80000
Missing values handled: 0
Final Train shape: (64000, 4) Target: (64000,)
Final Test shape: (16000, 4) Target: (16000,)


In [10]:
# Train models and generate PKLs
if 'X_train' in locals():
    models, results = train_and_evaluate_models(X_train, y_train, X_test, y_test, config['task_type'].lower())
    print("Model Results:")
    for name, metrics in results.items():
        print(f"{name}: {metrics}")
    
    # Generate and save PKLs for all models
    for name, model in models.items():
        bundle = {
            'model': model,
            'scaler': scaler,
            'encoders': encoders,
            'task_type': config['task_type'].lower(),
            'feature_columns': config['X_cols'],
            'target_column': config['y_col']
        }
        pkl_path = os.path.join(session_dir, f"{name}.pkl")
        with open(pkl_path, 'wb') as f:
            pickle.dump(bundle, f)
        print(f"✅ PKL saved for {name} at {pkl_path}")

Model Results:
LogisticRegression: {'Accuracy': 0.5176875, 'F1 Score': 0.4421735455711388}
DecisionTree: {'Accuracy': 0.5176875, 'F1 Score': 0.4421735455711388}
RandomForest: {'Accuracy': 0.5176875, 'F1 Score': 0.4421735455711388}
SVC: {'Accuracy': 0.5176875, 'F1 Score': 0.4421735455711388}
✅ PKL saved for LogisticRegression at session_20250927_203811/LogisticRegression.pkl
✅ PKL saved for DecisionTree at session_20250927_203811/DecisionTree.pkl
✅ PKL saved for RandomForest at session_20250927_203811/RandomForest.pkl
✅ PKL saved for SVC at session_20250927_203811/SVC.pkl


In [11]:
# Overwrite pipeline.py (as in original)
with open("pipeline.py", "w") as f:
    f.write("""
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

def preprocess_data(df, X_cols, y_col, task_type):
    report = {}
    encoders = {}
    missing_before = df.isnull().sum().sum()
    X = df[X_cols].copy()
    y = df[y_col].copy()

    # Handle missing values in X
    for col in X.columns:
        if X[col].dtype in ["int64", "float64"]:
            X[col] = X[col].fillna(X[col].mean())
        else:
            X[col] = X[col].fillna(X[col].mode()[0])

    # Handle missing values in y
    if y.isnull().sum() > 0:
        if y.dtype in ["int64", "float64"]:
            y = y.fillna(y.mean())
        else:
            y = y.fillna(y.mode()[0])

    # Encode categorical X columns
    for col in X.columns:
        if X[col].dtype == "object" or str(X[col].dtype) == "category":
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col].astype(str))
            encoders[col] = le

    # Encode target if classification
    if task_type == "classification" and (
        y.dtype == "object" or str(y.dtype) == "category"
    ):
        le = LabelEncoder()
        y = le.fit_transform(y.astype(str))
        encoders[y_col] = le

    # Scale features
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Calculate missing after
    missing_after = pd.concat([pd.DataFrame(X), pd.Series(y)], axis=1).isnull().sum().sum()
    report["rows_before"] = df.shape[0]
    report["rows_after"] = X_train.shape[0] + X_test.shape[0]
    report["missing_handled"] = missing_before - missing_after
    report["train_shape"] = X_train.shape
    report["test_shape"] = X_test.shape
    report["encoded_columns"] = list(encoders.keys())

    return X_train, X_test, y_train, y_test, report, encoders, scaler

def train_and_evaluate_models(X_train, y_train, X_test, y_test, task_type):
    models = {}
    results = {}
    if task_type == "classification":
        model_list = {
            'LogisticRegression': LogisticRegression(random_state=42),
            'DecisionTree': DecisionTreeClassifier(random_state=42),
            'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
            'SVC': SVC(random_state=42)
        }
    else:  # regression
        model_list = {
            'LinearRegression': LinearRegression(),
            'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42)
        }

    for name, model in model_list.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        if task_type == "classification":
            results[name] = {
                'Accuracy': accuracy_score(y_test, preds),
                'F1 Score': f1_score(y_test, preds, average='weighted')
            }
        else:
            results[name] = {
                'MSE': mean_squared_error(y_test, preds),
                'R2': r2_score(y_test, preds)
            }
        models[name] = model  # Store fitted model

    return models, results
""")
print("✅ pipeline.py updated")

✅ pipeline.py updated
